<a href="https://colab.research.google.com/github/Oruntu-Tanima-Proje/otProje/blob/main/01_veriHazirlik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📁 01 - Veri Hazırlık

## Domates Yaprak Hastalık Tanı Destek Sistemi

Bu notebook, **PlantVillage New Plant Diseases Dataset (Augmented)** veri setinden
domates alt kümesini ayıklayıp eğitim/doğrulama/test kümelerine ayırır.

### Veri Seti Bilgileri
- **Kaynak**: [Kaggle - New Plant Diseases Dataset](https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset)
- **Sınıflar**: 10 (9 hastalık + sağlıklı)
- **Toplam Görüntü**: ~22.000 (filtrelenmiş)
- **Görüntü Boyutu**: 256x256 RGB

### Ön Koşullar
- Google Colab hesabı
- Kaggle API anahtarı (`kaggle.json`)
- Google Drive bağlantısı (yedekleme için)


In [ ]:
# ============================================================
# 1. KAGGLE API KURULUMU
# ============================================================

# Kaggle kütüphanesini yükle
!pip install -q kaggle

# kaggle.json dosyasını yükle (sizden istenecek)
from google.colab import files
print("Lütfen kaggle.json dosyanızı yükleyin:")
files.upload()

# Kaggle config
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\n✅ Kaggle API hazır")

In [ ]:
# ============================================================
# 2. VERİ SETİNİ İNDİR VE ÇIKAR
# ============================================================

# Veri setini indir (~3 GB, 3-7 dakika sürer)
print("Veri seti indiriliyor...\n")
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset

# Çıkar
print("\nÇıkarılıyor...")
!unzip -q new-plant-diseases-dataset.zip -d plant_data/

# Zip'i sil (yer açmak için)
!rm new-plant-diseases-dataset.zip

print("\n✅ İndirme ve çıkarma tamamlandı")

In [ ]:
# ============================================================
# 3. DOMATES SINIFLARINI AYIKLA
# ============================================================

import os
import shutil

# Çift kopya ve etiketsiz test klasörünü temizle
for unwanted in [
    "plant_data/new plant diseases dataset(augmented)",
    "plant_data/test"
]:
    if os.path.exists(unwanted):
        shutil.rmtree(unwanted)

# Orijinal yollar
original_train = "plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
original_valid = "plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"

# Hedef yollar (sadece domates)
tomato_dir = "tomato_data"
tomato_train = f"{tomato_dir}/train"
tomato_valid = f"{tomato_dir}/valid"

# Domates sınıflarını bul (Tomato___ ile başlayanlar)
all_classes = sorted(os.listdir(original_train))
tomato_classes = [c for c in all_classes if c.startswith("Tomato")]

print(f"Toplam sınıf: {len(all_classes)} | Domates: {len(tomato_classes)}\n")
print("🍅 Domates sınıfları:")
for i, cls in enumerate(tomato_classes, 1):
    short = cls.replace("Tomato___", "").replace("_", " ")
    print(f"  {i:2}. {short}")

# Hedef klasörleri oluştur
os.makedirs(tomato_train, exist_ok=True)
os.makedirs(tomato_valid, exist_ok=True)

# Domates klasörlerini taşı (move ile yer kazan)
print("\nDomates klasörleri taşınıyor...")
for cls in tomato_classes:
    # Train
    src = os.path.join(original_train, cls)
    dst = os.path.join(tomato_train, cls)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)
    # Valid
    src = os.path.join(original_valid, cls)
    dst = os.path.join(tomato_valid, cls)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)

# Geri kalan veriyi sil (yer aç)
shutil.rmtree("plant_data")
print("\n✅ Domates filtreleme tamamlandı")

In [ ]:
# ============================================================
# 4. VALID'İ VALID + TEST OLARAK BÖL (50/50)
# ============================================================

import random

random.seed(42)  # Tekrarlanabilirlik

tomato_test = f"{tomato_dir}/test"
os.makedirs(tomato_test, exist_ok=True)

classes = sorted(os.listdir(tomato_valid))

print("Valid → Valid + Test ayrımı yapılıyor (50/50)...\n")
for cls in classes:
    valid_dir = os.path.join(tomato_valid, cls)
    test_dir = os.path.join(tomato_test, cls)
    os.makedirs(test_dir, exist_ok=True)

    images = os.listdir(valid_dir)
    random.shuffle(images)
    test_count = len(images) // 2
    for img in images[:test_count]:
        shutil.move(
            os.path.join(valid_dir, img),
            os.path.join(test_dir, img)
        )

# Final özet
print("\n" + "=" * 60)
print("🍅 FİNAL VERİ DAĞILIMI")
print("=" * 60)

total_grand = 0
for split, path in [("Train", tomato_train), ("Valid", tomato_valid), ("Test", tomato_test)]:
    total = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
    total_grand += total
    print(f"  {split:10s} {total:>6} görüntü")

print(f"  {'-' * 30}")
print(f"  {'TOPLAM':10s} {total_grand:>6} görüntü, {len(classes)} sınıf")
print("=" * 60)

In [ ]:
# ============================================================
# 5. VERİ SETİNİ DRIVE'A YEDEKLE
# (Sonraki notebook'larda yeniden indirmeye gerek kalmasın)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

drive_proje = "/content/drive/MyDrive/Domates_Projesi"
drive_data = f"{drive_proje}/data"

if not os.path.exists(drive_data):
    print("📦 Veri seti Drive'a yedekleniyor (~5 dakika)...")
    shutil.copytree("tomato_data", drive_data)
    print("\n✅ Yedekleme tamamlandı!")
else:
    print("✅ Drive'da yedek zaten var")

print(f"\nYedek konumu: {drive_data}")
print("\n📌 Sonraki notebook'lar bu yedekten veri kopyalayacak.")

## ✅ Veri Hazırlık Tamamlandı

### Çıktılar
- `tomato_data/train/` — Eğitim seti (~18.300 görüntü)
- `tomato_data/valid/` — Doğrulama seti (~2.300 görüntü)
- `tomato_data/test/` — Test seti (~2.300 görüntü)
- Drive'da yedek: `/content/drive/MyDrive/Domates_Projesi/data/`

### Sıradaki Adım
👉 `02_veri_kesfi.ipynb` notebook'unu açın ve sınıf dağılımı + örnek görüntüleri inceleyin.
